In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import ipaddress

In [2]:
data = pd.read_csv(r'C:\Users\vibee\cloud project\archive (10)\rba-dataset.csv', nrows=1000000)

In [3]:
data['Login Hour'] = pd.to_datetime(data['Login Timestamp']).dt.hour

In [4]:
data['Is Account Takeover'] = data['Is Account Takeover'].astype(np.uint8)
data['Is Attack IP'] = data['Is Attack IP'].astype(np.uint8)
data['Login Successful'] = data['Login Successful'].astype(np.uint8)

In [5]:
print(data.head())

   index          Login Timestamp              User ID  Round-Trip Time [ms]  \
0      0  2020-02-03 12:43:30.772 -4324475583306591935                   NaN   
1      1  2020-02-03 12:43:43.549 -4324475583306591935                   NaN   
2      2  2020-02-03 12:43:55.873 -3284137479262433373                   NaN   
3      3  2020-02-03 12:43:56.180 -4324475583306591935                   NaN   
4      4  2020-02-03 12:43:59.396 -4618854071942621186                   NaN   

      IP Address Country    Region       City     ASN  \
0    10.0.65.171      NO         -          -   29695   
1   194.87.207.6      AU         -          -   60117   
2  81.167.144.58      NO  Vestland  Urangsvag   29695   
3  170.39.78.152      US         -          -  393398   
4      10.0.0.47      US  Virginia    Ashburn  398986   

                                   User Agent String  \
0  Mozilla/5.0  (iPhone; CPU iPhone OS 13_4 like ...   
1  Mozilla/5.0  (Linux; Android 4.1; Galaxy Nexus...   
2  Mozil

In [6]:
data['Is Account Takeover'] = data['Is Account Takeover'].astype(np.uint8)
data['Is Attack IP'] = data['Is Attack IP'].astype(np.uint8)
data['Login Successful'] = data['Login Successful'].astype(np.uint8)

In [7]:
data = data.drop(columns=["Round-Trip Time [ms]", 'Region', 'City', 'Login Timestamp', 'index'])

In [8]:
data['User Agent String'], _ = pd.factorize(data['User Agent String'])
data['Browser Name and Version'], _ = pd.factorize(data['Browser Name and Version'])
data['OS Name and Version'], _ = pd.factorize(data['OS Name and Version'])

In [9]:
def ip_to_int(ip):
    return int(ipaddress.ip_address(ip))

data['IP Address'] = data['IP Address'].apply(ip_to_int)

In [10]:
categorical_cols = ['Country', 'Device Type']
numeric_cols = ['ASN', 'Login Hour', 'IP Address', 'User Agent String', 'Browser Name and Version', 'OS Name and Version']

In [11]:
features = data.drop(['Is Attack IP', 'Is Account Takeover'], axis=1)
labels = data['Is Account Takeover']

X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

In [12]:
# Preprocessors
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(), categorical_cols)
    ])

In [13]:
# Classifiers
classifiers = {
    'logistic_regression': LogisticRegression(max_iter=1000),
    'decision_tree': DecisionTreeClassifier(),
    'svm': SVC(probability=True),
    'random_forest': RandomForestClassifier()
}

In [14]:

# A function to choose classifiers
def make_pipeline(classifier_key):
    if classifier_key in classifiers:
        clf = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', classifiers[classifier_key])
        ])
        return clf
    else:
        raise ValueError(f"Classifier {classifier_key} is not defined")

LOGISTIC REGRESSION

In [15]:
classifier_key = 'logistic_regression'
pipeline = make_pipeline(classifier_key)
pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [16]:
# Evaluation
predictions = pipeline.predict(X_test)
probs = pipeline.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, probs)

print(f"AUC Score: {auc_score}")

AUC Score: 0.9511676008473461


DECISION TREE

In [17]:
classifier_key = 'decision_tree'
pipeline = make_pipeline(classifier_key)
pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [18]:
# Evaluation
predictions = pipeline.predict(X_test)
probs = pipeline.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, probs)

print(f"AUC Score: {auc_score}")

AUC Score: 0.8333333333333333


Support Vector Machines (SVMs)

In [19]:
classifier_key = 'svm'
pipeline = make_pipeline(classifier_key)
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [20]:
# Evaluation
predictions = pipeline.predict(X_test)
probs = pipeline.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, probs)

print(f"AUC Score: {auc_score}")


AUC Score: 0.9879314856389512


RANDOM FOREST

In [21]:
classifier_key = 'random_forest'
pipeline = make_pipeline(classifier_key)
pipeline.fit(X_train, y_train)

# Evaluation
predictions = pipeline.predict(X_test)
probs = pipeline.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, probs)

print(f"AUC Score: {auc_score}")

AUC Score: 0.8333033328833266


ANOMOLY LOGIN ACTIVITIES

In [22]:
data = pd.read_csv('C:/Users/vibee/cloud project/archive (10)/rba-dataset.csv')

In [23]:
data.head()

,index,Login Timestamp,User ID,Round-Trip Time [ms],IP Address,Country,Region,City,ASN,User Agent String,Browser Name and Version,OS Name and Version,Device Type,Login Successful,Is Attack IP,Is Account Takeover
0,0,2020-02-03 12:43:30.772,-4324475583306591935,NaN,10.0.65.171,NO,-,-,29695,Mozilla/5.0 (iPhone; CPU iPhone OS 13_4 like ...,Firefox 20.0.0.1618,iOS 13.4,mobile,False,False,False
1,1,2020-02-03 12:43:43.549,-4324475583306591935,NaN,194.87.207.6,AU,-,-,60117,Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus...,Chrome Mobile 46.0.2490,Android 4.1,mobile,False,False,False
2,2,2020-02-03 12:43:55.873,-3284137479262433373,NaN,81.167.144.58,NO,Vestland,Urangsvag,29695,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,iOS 7.1,mobile,True,False,False
3,3,2020-02-03 12:43:56.180,-4324475583306591935,NaN,170.39.78.152,US,-,-,393398,Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus...,Chrome Mobile WebView 85.0.4183,Android 4.1,mobile,False,False,False
4,4,2020-02-03 12:43:59.396,-4618854071942621186,NaN,10.0.0.47,US,Virginia,Ashburn,398986,Mozilla/5.0 (Linux; U; Android 2.2) Build/NMA...,Chrome Mobile WebView 85.0.4183,Android 2.2,mobile,False,True,False


In [24]:
data.columns

Index(['index', 'Login Timestamp', 'User ID', 'Round-Trip Time [ms]',
       'IP Address', 'Country', 'Region', 'City', 'ASN', 'User Agent String',
       'Browser Name and Version', 'OS Name and Version', 'Device Type',
       'Login Successful', 'Is Attack IP', 'Is Account Takeover'],
      dtype='object')

In [25]:
data.dtypes

index                         int64
Login Timestamp              object
User ID                       int64
Round-Trip Time [ms]        float64
IP Address                   object
Country                      object
Region                       object
City                         object
ASN                           int64
User Agent String            object
Browser Name and Version     object
OS Name and Version          object
Device Type                  object
Login Successful               bool
Is Attack IP                   bool
Is Account Takeover            bool
dtype: object

In [26]:
data.isna().mean().sort_values(ascending=False).head(10)

Round-Trip Time [ms]    0.959195
Region                  0.001516
City                    0.000275
Device Type             0.000049
User ID                 0.000000
Login Timestamp         0.000000
Country                 0.000000
index                   0.000000
IP Address              0.000000
ASN                     0.000000
dtype: float64

In [27]:
data.shape

(31269264, 16)

In [28]:

# Convert timestamp
data["Login Timestamp"] = pd.to_datetime(data["Login Timestamp"], utc=True)

# Extract date and hour
data["date"] = data["Login Timestamp"].dt.date
data["hour"] = data["Login Timestamp"].dt.hour

# Create night login feature (IMPORTANT FIX)
data["is_night"] = data["hour"].apply(lambda x: 1 if x < 6 or x >= 22 else 0)

# Now aggregate
daily = data.groupby(["User ID", "date"]).agg(
    login_count=("Login Timestamp", "count"),
    success_rate=("Login Successful", "mean"),
    night_login_ratio=("is_night", "mean"),
    unique_ips=("IP Address", "nunique"),
    unique_countries=("Country", "nunique"),
    unique_asns=("ASN", "nunique"),
    unique_devices=("Device Type", "nunique"),
).reset_index()

print(daily.head())
print(daily.shape)

               User ID        date  login_count  success_rate  \
0 -9223371191532286299  2021-01-12            1           0.0   
1 -9223369357534132497  2020-09-13            1           1.0   
2 -9223369089733265380  2020-04-22            1           0.0   
3 -9223360723444354188  2020-04-11            1           1.0   
4 -9223360723444354188  2020-07-15            1           1.0   

   night_login_ratio  unique_ips  unique_countries  unique_asns  \
0                0.0           1                 1            1   
1                0.0           1                 1            1   
2                0.0           1                 1            1   
3                1.0           1                 1            1   
4                1.0           1                 1            1   

   unique_devices  
0               1  
1               1  
2               1  
3               1  
4               1  
(12090619, 9)


In [29]:
# per-user behavioral baselines
user_baseline = daily.groupby("User ID").agg(
    mean_logins=("login_count", "mean"),
    std_logins=("login_count", "std"),
    mean_night_ratio=("night_login_ratio", "mean"),
    mean_unique_ips=("unique_ips", "mean"),
    mean_unique_countries=("unique_countries", "mean"),
    mean_unique_devices=("unique_devices", "mean"),
).reset_index()

user_baseline.head(), user_baseline.shape

(               User ID  mean_logins  std_logins  mean_night_ratio  \
 0 -9223371191532286299         1.00         NaN               0.0   
 1 -9223369357534132497         1.00         NaN               0.0   
 2 -9223369089733265380         1.00         NaN               0.0   
 3 -9223360723444354188         1.75    0.957427               0.5   
 4 -9223358650992576877         1.00         NaN               1.0   
 
    mean_unique_ips  mean_unique_countries  mean_unique_devices  
 0              1.0                    1.0                  1.0  
 1              1.0                    1.0                  1.0  
 2              1.0                    1.0                  1.0  
 3              1.0                    1.0                  1.0  
 4              1.0                    1.0                  1.0  ,
 (4304857, 7))

In [30]:
# join baselines back to daily data
daily_with_base = daily.merge(user_baseline, on="User ID", how="left")

In [31]:
# handle missing std (users with only 1 day of history)
daily_with_base["std_logins"] = daily_with_base["std_logins"].fillna(1.0)


In [32]:
# deviation features
daily_with_base["z_logins"] = (
    daily_with_base["login_count"] - daily_with_base["mean_logins"]
) / daily_with_base["std_logins"]

daily_with_base["ip_ratio"] = (
    daily_with_base["unique_ips"] / daily_with_base["mean_unique_ips"]
)

daily_with_base["country_ratio"] = (
    daily_with_base["unique_countries"] / daily_with_base["mean_unique_countries"]
)

daily_with_base["device_ratio"] = (
    daily_with_base["unique_devices"] / daily_with_base["mean_unique_devices"]
)

daily_with_base["night_deviation"] = (
    daily_with_base["night_login_ratio"] - daily_with_base["mean_night_ratio"]
)

daily_with_base[
    ["z_logins", "ip_ratio", "country_ratio", "device_ratio", "night_deviation"]
].describe()

,z_logins,ip_ratio,country_ratio,device_ratio,night_deviation
count,9.299874e+06,1.209062e+07,1.209062e+07,1.209058e+07,1.209062e+07
mean,2.580874e-18,1.000000e+00,1.000000e+00,1.000000e+00,2.263890e-18
std,7.927150e-01,1.445405e-01,2.708991e-02,1.235195e-01,2.231882e-01
min,-1.472870e+01,1.666667e-01,4.000000e-01,0.000000e+00,-9.682540e-01
25%,-4.830459e-01,1.000000e+00,1.000000e+00,1.000000e+00,-4.000000e-02
50%,-2.672612e-01,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00
75%,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00
max,1.263950e+01,9.492228e+00,4.107143e+00,3.360000e+00,9.970060e-01


In [33]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# select deviation features
features = [
    "z_logins",
    "ip_ratio",
    "country_ratio",
    "device_ratio",
    "night_deviation"
]

X = daily_with_base[features].replace([np.inf, -np.inf], np.nan).fillna(0)

# scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# train isolation forest
iso = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42,
    n_jobs=-1
)

iso.fit(X_scaled)

# Anomaly scores (higher = more anomalous)
daily_with_base["anomaly_score"] = -iso.decision_function(X_scaled)

print(daily_with_base["anomaly_score"].describe())

count    1.209062e+07
mean    -2.864268e-01
std      8.054054e-02
min     -3.463732e-01
25%     -3.463732e-01
50%     -3.218753e-01
75%     -2.547018e-01
max      1.363228e-01
Name: anomaly_score, dtype: float64


In [34]:
# rank most anomalous user-days
alerts = daily_with_base.sort_values("anomaly_score", ascending=False)

alerts_cols = [
    "User ID", "date", "anomaly_score",
    "login_count", "z_logins",
    "unique_ips", "ip_ratio",
    "unique_countries", "country_ratio",
    "unique_devices", "device_ratio",
    "night_login_ratio", "night_deviation"
]

alerts[alerts_cols].head(20)

,User ID,date,anomaly_score,login_count,z_logins,unique_ips,ip_ratio,unique_countries,country_ratio,unique_devices,device_ratio,night_login_ratio,night_deviation
5436045,-915874203850528192,2020-08-03,0.136323,7,4.245453,3,2.464286,1,0.884615,2,1.642857,0.000000,-0.217391
11444829,8234718033283260712,2021-02-26,0.133945,7,4.003515,3,2.744681,2,1.911111,2,1.829787,0.000000,-0.511628
5313145,-1105875135061296654,2020-06-09,0.133644,7,3.931233,5,3.431373,1,0.921053,2,1.627907,0.000000,-0.352381
11359653,8105329987683138826,2020-08-04,0.133377,7,5.678123,5,4.075630,1,0.950980,2,1.940000,0.142857,-0.228031
5277673,-1159271285732371893,2020-05-31,0.132781,6,9.623764,3,2.914286,2,1.961538,2,1.980583,0.166667,-0.300654
11008659,7566036960400292984,2020-02-15,0.132358,3,4.192576,3,2.896552,1,0.988235,2,1.976471,0.333333,-0.634921
10946463,7470558100220665624,2020-07-24,0.131830,9,8.128390,4,3.072072,2,1.988338,2,1.965418,0.555556,0.378706
11359627,8105329987683138826,2020-06-26,0.131508,6,4.656271,4,3.260504,1,0.950980,2,1.940000,0.166667,-0.204222
11986533,9062384034431216043,2020-03-10,0.131241,14,3.443461,3,2.250000,1,0.937500,2,1.875000,0.000000,-0.346667
3584149,-3755286854393492752,2020-12-09,0.131099,4,5.150186,3,2.861538,2,1.952756,2,1.952756,0.000000,-0.262097


In [35]:
# attach Attack take over labels at user-day level
labels = (
    data.assign(date=data["Login Timestamp"].dt.date)
    .groupby(["User ID", "date"])
    .agg(
        ato=("Is Account Takeover", "max"),
        attack_ip=("Is Attack IP", "max")
    )
    .reset_index()
)

alerts_with_labels = alerts.merge(
    labels,
    on=["User ID", "date"],
    how="left"
)

alerts_with_labels[["anomaly_score", "ato", "attack_ip"]].head(20)

,anomaly_score,ato,attack_ip
0,0.136323,False,True
1,0.133945,False,False
2,0.133644,False,True
3,0.133377,False,False
4,0.132781,False,False
5,0.132358,False,False
6,0.131830,True,True
7,0.131508,False,False
8,0.131241,False,True
9,0.131099,False,False


In [36]:
top_1pct = alerts_with_labels.head(int(0.01 * len(alerts_with_labels)))

top_1pct["ato"].mean(), top_1pct["attack_ip"].mean()

(np.float64(0.0001406050981754421), np.float64(0.06034439978164855))

In [37]:
# Baseline ATO rate in entire dataset
baseline_ato_rate = labels["ato"].mean()

baseline_ato_rate

np.float64(1.149651643145814e-05)

In [38]:
top_01pct = alerts_with_labels.head(int(0.001 * len(alerts_with_labels)))
top_01pct["ato"].mean()

np.float64(0.00024813895781637717)

In [39]:
lift = top_01pct["ato"].mean() / baseline_ato_rate
lift

np.float64(21.583838834639486)

In [40]:
def explain_alert(row):
    reasons = []
    if row["z_logins"] > 3:
        reasons.append("login volume far above baseline")
    if row["ip_ratio"] > 2:
        reasons.append("multiple new IP addresses")
    if row["country_ratio"] > 1.5:
        reasons.append("logins from multiple countries")
    if row["device_ratio"] > 1.5:
        reasons.append("new devices observed")
    if row["night_deviation"] > 0.5:
        reasons.append("unusual nighttime activity")
    return "; ".join(reasons)

alerts_with_labels["alert_reason"] = alerts_with_labels.apply(explain_alert, axis=1)
alerts_with_labels[["anomaly_score", "alert_reason"]].head(10)

,anomaly_score,alert_reason
0,0.136323,login volume far above baseline; multiple new ...
1,0.133945,login volume far above baseline; multiple new ...
2,0.133644,login volume far above baseline; multiple new ...
3,0.133377,login volume far above baseline; multiple new ...
4,0.132781,login volume far above baseline; multiple new ...
5,0.132358,login volume far above baseline; multiple new ...
6,0.131830,login volume far above baseline; multiple new ...
7,0.131508,login volume far above baseline; multiple new ...
8,0.131241,login volume far above baseline; multiple new ...
9,0.131099,login volume far above baseline; multiple new ...


In [41]:
alerts_with_labels[["anomaly_score","alert_reason","ato","attack_ip"]].head(20)

,anomaly_score,alert_reason,ato,attack_ip
0,0.136323,login volume far above baseline; multiple new ...,False,True
1,0.133945,login volume far above baseline; multiple new ...,False,False
2,0.133644,login volume far above baseline; multiple new ...,False,True
3,0.133377,login volume far above baseline; multiple new ...,False,False
4,0.132781,login volume far above baseline; multiple new ...,False,False
5,0.132358,login volume far above baseline; multiple new ...,False,False
6,0.131830,login volume far above baseline; multiple new ...,True,True
7,0.131508,login volume far above baseline; multiple new ...,False,False
8,0.131241,login volume far above baseline; multiple new ...,False,True
9,0.131099,login volume far above baseline; multiple new ...,False,False


In [42]:
top_alerts = alerts_with_labels.sort_values("anomaly_score", ascending=False).head(500)
top_alerts.to_csv("top_alerts.csv", index=False)

In [43]:
top_alerts = alerts_with_labels.sort_values("anomaly_score", ascending=False).head(500)

top_alerts.to_csv(
    r"C:\Users\vibee\cloud project\top_alerts.csv",
    index=False
)